# Chapter 8: Gaussian, mean and principal curvatures

Source orientation: printed pages 179-214; PDF pages 183-217; sections 8.1-8.6.

This notebook is an original standalone teaching version of the assigned span. The source was used only to orient the chapter sequence and terminology. No textbook prose, exercise text, figures, screenshots, page crops, or page layouts are reproduced here.

## Chapter Question

How much of the local and global shape of a surface can be read from the shape operator?

The chapter answer has three levels. At one point, the shape operator is a self-adjoint linear map on the tangent plane. Its determinant, half-trace, and eigenvalues give Gaussian curvature, mean curvature, and principal curvatures. Across a surface, those numbers sort local models into elliptic, hyperbolic, parabolic, planar, flat, constant-K, and constant-H behavior. On a compact surface, the local numbers cannot all avoid positive Gaussian curvature.

## Computational Translation Guide

| Book-side idea | Computational translation |
| --- | --- |
| Shape operator $W$ | A symmetric $2 \times 2$ matrix after choosing an orthonormal tangent frame. |
| Gaussian curvature $K$ | `det(W)`, or the product of the two eigenvalues of $W$. |
| Mean curvature $H$ | `trace(W) / 2`, or the arithmetic mean of the two eigenvalues. |
| Principal curvatures and directions | Eigenvalues and orthonormal eigenvectors of the self-adjoint map $W$. |
| Normal curvature in a tangent direction | Quadratic form $\kappa_n(\theta) = \kappa_1\cos^2\theta + \kappa_2\sin^2\theta$. |
| Constant Gaussian curvature | A surface family where the computed $K$ is the same at every sampled point; for a surface of revolution with arc-length meridian this becomes $K = -f''/f$. |
| Flat surface | A surface with $K=0$, so at least one principal curvature is zero away from planar or umbilic exceptions. |
| Constant mean curvature | A surface family where $H$ is fixed; parallel surfaces transform principal curvatures by a rational formula. |
| Compact-surface constraint | A support-sphere argument: a farthest point from the origin forces both principal curvatures to have the same sign. |

## Route Through The Notebook

1. Build $K$, $H$, and the principal curvatures directly from a shape-operator matrix.
2. Use Euler's normal-curvature formula to see principal directions as extrema.
3. Draw local second-order surface models for curvature sign.
4. Compare constant Gaussian curvature models with both a symbolic check and an interactive 3D artifact.
5. Contrast flatness with being planar by looking at developable surfaces.
6. Use parallel-surface formulas to connect constant mean curvature with constant positive Gaussian curvature.
7. Visualize the compact-surface support-sphere argument and save a final sanity ledger.

In [ ]:
from __future__ import annotations

import json
import math
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import sympy as sp

BOOK_ROOT = Path.cwd().resolve()
for candidate in [BOOK_ROOT, *BOOK_ROOT.parents]:
    if (candidate / "00-book-index.ipynb").exists() and (candidate / "utils").exists():
        BOOK_ROOT = candidate
        break
else:
    raise RuntimeError("Could not find the Pressley book root")

if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import assert_artifact, artifact_path, display_artifact, save_json, save_matplotlib

UNIT = "chapter-08"
ARTIFACT_ROOT = BOOK_ROOT / "artifacts"
UNIT_ARTIFACT_ROOT = ARTIFACT_ROOT / UNIT
for subfolder in ["figures", "interactive", "checks", "tables"]:
    (UNIT_ARTIFACT_ROOT / subfolder).mkdir(parents=True, exist_ok=True)

SOURCE_SPAN = {
    "book": "Andrew Pressley, Elementary Differential Geometry, Second Edition",
    "chapter": "Chapter 8: Gaussian, mean and principal curvatures",
    "printed_pages": "179-214",
    "pdf_pages": "183-217",
    "sections": "8.1-8.6",
    "source_use": "orientation only; notebook prose, code, and visuals are original",
}
source_span_path = save_json(SOURCE_SPAN, UNIT, "checks", "source-span.json", root=ARTIFACT_ROOT)
source_span_path

## 1. From The Shape Operator To $K$, $H$, And Principal Curvatures

At a fixed point of an oriented surface, all of this chapter begins with one linear map: the shape operator $W$ on the tangent plane. In an orthonormal tangent frame, self-adjointness means we can treat $W$ as a symmetric matrix. The determinant and trace are independent of the chosen orthonormal frame, and the eigenvalues are the principal curvatures.

The symbolic check below keeps the algebra visible. If $\kappa_1$ and $\kappa_2$ are the eigenvalues, then

$$K = \kappa_1\kappa_2,\qquad H = \frac{\kappa_1 + \kappa_2}{2},\qquad x^2 - 2Hx + K = 0.$$

The last equation is the characteristic equation written in curvature language.

In [ ]:
a, b, c, k1, k2 = sp.symbols("a b c k1 k2", real=True)
lam = sp.symbols("lam")
W = sp.Matrix([[a, b], [b, c]])
K_expr = sp.factor(W.det())
H_expr = sp.factor(sp.trace(W) / 2)
char_expr = sp.factor(W.charpoly(lam).as_expr())
curvature_char_expr = sp.factor(lam**2 - 2 * H_expr * lam + K_expr)

identity_checks = {
    "characteristic_matches_curvature_polynomial": sp.expand(char_expr - curvature_char_expr) == 0,
    "eigenvalue_discriminant_identity": sp.expand(((k1 + k2) / 2) ** 2 - k1 * k2 - ((k1 - k2) / 2) ** 2) == 0,
}
assert all(identity_checks.values())

sample_operators = [
    {"surface_model": "unit sphere, outward normal", "matrix": [[1.0, 0.0], [0.0, 1.0]]},
    {"surface_model": "cylinder of radius 2", "matrix": [[0.0, 0.0], [0.0, 0.5]]},
    {"surface_model": "saddle z=(x^2-y^2)/2 at the origin", "matrix": [[1.0, 0.0], [0.0, -1.0]]},
    {"surface_model": "elliptic graph z=(2x^2+y^2)/2 at the origin", "matrix": [[2.0, 0.0], [0.0, 1.0]]},
]

rows = []
for item in sample_operators:
    M = np.array(item["matrix"], dtype=float)
    eigenvalues = np.linalg.eigvalsh(M)
    rows.append(
        {
            "surface_model": item["surface_model"],
            "kappa_1": float(eigenvalues[0]),
            "kappa_2": float(eigenvalues[1]),
            "K_det": float(np.linalg.det(M)),
            "H_half_trace": float(np.trace(M) / 2.0),
        }
    )

shape_table = pd.DataFrame(rows)
shape_checks = {
    "symbolic": {name: bool(value) for name, value in identity_checks.items()},
    "characteristic_polynomial": str(curvature_char_expr),
    "samples": rows,
    "max_det_product_residual": float(
        max(abs(row["K_det"] - row["kappa_1"] * row["kappa_2"]) for row in rows)
    ),
    "max_trace_mean_residual": float(
        max(abs(row["H_half_trace"] - 0.5 * (row["kappa_1"] + row["kappa_2"])) for row in rows)
    ),
}
shape_checks_path = save_json(shape_checks, UNIT, "checks", "shape-operator-invariants.json", root=ARTIFACT_ROOT)
shape_table

## 2. Principal Directions As Normal-Curvature Extrema

Euler's normal-curvature formula turns a linear-algebra fact into a visible surface fact. Once the tangent basis has been rotated into principal directions, the normal curvature in a unit direction making angle $\theta$ with the first principal direction is

$$\kappa_n(\theta) = \kappa_1\cos^2\theta + \kappa_2\sin^2\theta.$$

The figure below is not a surface picture yet. It is the tangent-plane measuring device. Inspect the black and gray axes: normal curvature reaches its maximum and minimum exactly along the principal directions, except at an umbilic where every direction ties.

In [ ]:
def normal_curvature(theta, kappa_1, kappa_2):
    return kappa_1 * np.cos(theta) ** 2 + kappa_2 * np.sin(theta) ** 2


principal_cases = [
    ("elliptic", 2.0, 0.6),
    ("hyperbolic", 1.4, -0.8),
    ("parabolic", 1.2, 0.0),
    ("umbilic", 1.0, 1.0),
]

theta = np.linspace(0, 2 * np.pi, 2001)
fig, axes = plt.subplots(2, 2, figsize=(11, 8), subplot_kw={"projection": "polar"})
principal_checks = []
for ax, (label, ka, kb) in zip(axes.ravel(), principal_cases):
    values = normal_curvature(theta, ka, kb)
    radius = np.abs(values)
    colors = np.where(values >= 0, "#2563eb", "#dc2626")
    ax.scatter(theta, radius, c=colors, s=4, alpha=0.75)
    for angle, color, axis_label in [(0, "#111827", "t1"), (np.pi / 2, "#6b7280", "t2")]:
        ax.plot([angle, angle], [0, max(radius) * 1.08], color=color, lw=2)
        ax.text(angle, max(radius) * 1.16, axis_label, ha="center", va="center", fontsize=10)
    ax.set_title(f"{label}: k1={ka:g}, k2={kb:g}", va="bottom")
    ax.set_yticklabels([])
    principal_checks.append(
        {
            "case": label,
            "kappa_1": ka,
            "kappa_2": kb,
            "sample_min": float(values.min()),
            "sample_max": float(values.max()),
            "expected_min": float(min(ka, kb)),
            "expected_max": float(max(ka, kb)),
            "extrema_residual": float(
                max(abs(values.min() - min(ka, kb)), abs(values.max() - max(ka, kb)))
            ),
        }
    )
fig.suptitle("Euler normal curvature: principal directions are extrema", y=0.98, fontsize=14)
principal_path = save_matplotlib(fig, UNIT, "figures", "principal-chapter-08.png", root=ARTIFACT_ROOT)
plt.close(fig)
principal_checks_path = save_json(
    {"normal_curvature_extrema": principal_checks},
    UNIT,
    "checks",
    "principal-direction-checks.json",
    root=ARTIFACT_ROOT,
)
display_artifact(principal_path, width=900)

## 3. Curvature Sign In Local 3D Models

Near a non-umbilic point, principal coordinates make the second-order surface model look like

$$z = \frac{1}{2}(\kappa_1 x^2 + \kappa_2 y^2).$$

This approximation is enough to see why $K=\kappa_1\kappa_2$ controls the local type. Positive $K$ bends the same way in both principal directions. Negative $K$ bends in opposite directions. Zero $K$ leaves one principal direction flat to second order.

In [ ]:
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

xgrid = np.linspace(-1.2, 1.2, 55)
ygrid = np.linspace(-1.2, 1.2, 55)
X, Y = np.meshgrid(xgrid, ygrid)
local_models = [
    ("elliptic K>0", 1.2, 0.55, "#2563eb"),
    ("hyperbolic K<0", 1.0, -0.75, "#dc2626"),
    ("parabolic K=0", 1.0, 0.0, "#d97706"),
    ("planar K=0", 0.0, 0.0, "#6b7280"),
]
fig = plt.figure(figsize=(13, 9))
curvature_sign_checks = []
for idx, (label, ka, kb, color) in enumerate(local_models, start=1):
    ax = fig.add_subplot(2, 2, idx, projection="3d")
    Z = 0.5 * (ka * X**2 + kb * Y**2)
    ax.plot_surface(X, Y, Z, rstride=1, cstride=1, linewidth=0, alpha=0.82, color=color, shade=True)
    ax.plot([-1.15, 1.15], [0, 0], [0.5 * ka * 1.15**2, 0.5 * ka * 1.15**2], color="#111827", lw=2)
    ax.plot([0, 0], [-1.15, 1.15], [0.5 * kb * 1.15**2, 0.5 * kb * 1.15**2], color="#f9fafb", lw=2)
    ax.set_title(f"{label}\nK={ka*kb:.2f}, H={(ka+kb)/2:.2f}")
    ax.set_xlabel("principal x")
    ax.set_ylabel("principal y")
    ax.set_zlabel("height")
    ax.view_init(elev=25, azim=-48)
    ax.set_box_aspect((1, 1, 0.7))
    curvature_sign_checks.append(
        {
            "case": label,
            "kappa_1": ka,
            "kappa_2": kb,
            "K": ka * kb,
            "H": 0.5 * (ka + kb),
            "sign_class": "positive" if ka * kb > 0 else "negative" if ka * kb < 0 else "zero",
        }
    )
fig.suptitle("Curvature sign from the quadratic surface model", fontsize=15)
fig.tight_layout()
curvature_sign_path = save_matplotlib(fig, UNIT, "figures", "curvature-sign-chapter-08.png", root=ARTIFACT_ROOT)
plt.close(fig)
curvature_sign_checks_path = save_json(
    {"local_quadratic_models": curvature_sign_checks},
    UNIT,
    "checks",
    "curvature-sign-checks.json",
    root=ARTIFACT_ROOT,
)
display_artifact(curvature_sign_path, width=920)

## 4. Constant Gaussian Curvature As A Surface-Of-Revolution ODE

For an arc-length parametrized meridian surface of revolution

$$\sigma(u,v) = (f(u)\cos v, f(u)\sin v, g(u)), \qquad f'(u)^2 + g'(u)^2 = 1,$$

one useful computational test is

$$K = -\frac{f''(u)}{f(u)}.$$

That turns constant Gaussian curvature into a one-dimensional ODE. The static plot checks the ODE for three radius functions. The interactive 3D artifact then lets the same comparison be rotated: a positive-K sphere band, a zero-K cylinder, and a negative-K pseudosphere-type local model built from $f(u)=e^u$ on an interval where the meridian remains regular.

In [ ]:
u = sp.symbols("u", real=True)
constant_k_models = [
    {"name": "sphere band", "f": sp.cos(u), "target_K": sp.Integer(1), "u_min": -1.1, "u_max": 1.1},
    {"name": "cylinder", "f": sp.Integer(1), "target_K": sp.Integer(0), "u_min": -1.2, "u_max": 1.2},
    {"name": "negative-K revolution", "f": sp.exp(u), "target_K": sp.Integer(-1), "u_min": -2.4, "u_max": -0.04},
]

constant_k_checks = []
fig, ax = plt.subplots(figsize=(10, 5.6))
for model in constant_k_models:
    K_symbolic = sp.simplify(-sp.diff(model["f"], u, 2) / model["f"])
    residual = sp.simplify(K_symbolic - model["target_K"])
    assert residual == 0
    us = np.linspace(model["u_min"], model["u_max"], 300)
    K_values = np.full_like(us, float(model["target_K"]))
    ax.plot(us, K_values, lw=2.5, label=f"{model['name']}: K={float(model['target_K']):g}")
    constant_k_checks.append(
        {
            "model": model["name"],
            "radius_function": str(model["f"]),
            "symbolic_K": str(K_symbolic),
            "target_K": float(model["target_K"]),
            "symbolic_residual_is_zero": bool(residual == 0),
        }
    )
ax.axhline(0, color="#111827", lw=1)
ax.set_xlabel("meridian parameter u")
ax.set_ylabel("computed K = -f''/f")
ax.set_title("Constant Gaussian curvature as a radius-function ODE")
ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5))
ax.grid(True, alpha=0.25)
constant_k_path = save_matplotlib(fig, UNIT, "figures", "constant-k-chapter-08.png", root=ARTIFACT_ROOT)
plt.close(fig)

v = np.linspace(0, 2 * np.pi, 96)
fig3d = go.Figure()
plot_specs = [
    ("sphere band, K=1", np.linspace(-1.1, 1.1, 70), lambda s: np.cos(s), lambda s: np.sin(s), -3.0, "Blues"),
    ("cylinder, K=0", np.linspace(-1.2, 1.2, 70), lambda s: np.ones_like(s), lambda s: s, 0.0, "Greys"),
]
us_neg = np.linspace(-2.4, -0.04, 95)
f_neg = np.exp(us_neg)
gprime_neg = np.sqrt(np.maximum(1 - f_neg**2, 0))
g_neg = np.zeros_like(us_neg)
g_neg[1:] = np.cumsum(0.5 * (gprime_neg[1:] + gprime_neg[:-1]) * np.diff(us_neg))
plot_specs.append(("negative K local model, K=-1", us_neg, lambda s, vals=f_neg: vals, lambda s, vals=g_neg: vals, 3.0, "Reds"))

for name, us, f_fun, g_fun, x_offset, colorscale in plot_specs:
    radius = f_fun(us)
    height = g_fun(us)
    V, U = np.meshgrid(v, us)
    R = radius[:, None]
    Z = height[:, None]
    X3 = x_offset + R * np.cos(V)
    Y3 = R * np.sin(V)
    Z3 = np.broadcast_to(Z, X3.shape)
    fig3d.add_trace(
        go.Surface(
            x=X3,
            y=Y3,
            z=Z3,
            colorscale=colorscale,
            showscale=False,
            opacity=0.92,
            name=name,
            hovertemplate=name + "<extra></extra>",
        )
    )
fig3d.update_layout(
    title="Constant Gaussian curvature comparison: K=1, K=0, K=-1",
    scene={"aspectmode": "data", "xaxis_title": "x", "yaxis_title": "y", "zaxis_title": "z"},
    margin={"l": 0, "r": 0, "t": 45, "b": 0},
)
constant_k_html = artifact_path(UNIT, "interactive", "constant-gaussian-curvature-surfaces.html", root=ARTIFACT_ROOT)
fig3d.write_html(str(constant_k_html), include_plotlyjs=True, full_html=True)
constant_k_checks_path = save_json(
    {"constant_gaussian_curvature_models": constant_k_checks, "interactive_artifact": constant_k_html.name},
    UNIT,
    "checks",
    "constant-gaussian-curvature.json",
    root=ARTIFACT_ROOT,
)
display_artifact(constant_k_path, width=900)
display_artifact(constant_k_html, width="100%", height=560)

## 5. Flat Does Not Mean Planar

The equation $K=0$ says the product of the principal curvatures is zero. A plane has both principal curvatures zero, but a cylinder, cone, or tangent developable can have one zero principal curvature and one nonzero principal curvature. That is why flat surfaces can be made from ruled developable pieces rather than only from open pieces of a plane.

Inspect the ruling lines in the next figure. Along each ruling direction the normal does not change to first order, so one principal curvature is zero. The surface can still bend in the transverse direction.

In [ ]:
fig = plt.figure(figsize=(14, 4.8))
flat_checks = []

ax = fig.add_subplot(1, 3, 1, projection="3d")
vv, zz = np.meshgrid(np.linspace(0, 2 * np.pi, 80), np.linspace(-1.2, 1.2, 35))
R = 1.0
Xc = R * np.cos(vv)
Yc = R * np.sin(vv)
Zc = zz
ax.plot_surface(Xc, Yc, Zc, color="#60a5fa", alpha=0.82, linewidth=0)
for angle in np.linspace(0, 2 * np.pi, 9)[:-1]:
    ax.plot([R * np.cos(angle)] * 2, [R * np.sin(angle)] * 2, [-1.25, 1.25], color="#111827", lw=1.1)
ax.set_title("cylinder: (0, 1/R)")
ax.set_box_aspect((1, 1, 1.3))
flat_checks.append({"surface": "cylinder", "principal_curvatures": [0.0, 1.0 / R], "K": 0.0})

ax = fig.add_subplot(1, 3, 2, projection="3d")
vv, rr = np.meshgrid(np.linspace(0, 2 * np.pi, 80), np.linspace(0.2, 1.25, 35))
Xcone = rr * np.cos(vv)
Ycone = rr * np.sin(vv)
Zcone = 1.25 - rr
ax.plot_surface(Xcone, Ycone, Zcone, color="#f59e0b", alpha=0.82, linewidth=0)
for angle in np.linspace(0, 2 * np.pi, 9)[:-1]:
    ax.plot([0, 1.25 * np.cos(angle)], [0, 1.25 * np.sin(angle)], [1.25, 0], color="#111827", lw=1.1)
ax.set_title("cone: one ruling curvature is 0")
ax.set_box_aspect((1, 1, 1.1))
flat_checks.append({"surface": "cone away from apex", "principal_curvatures": [0.0, "nonzero transverse"], "K": 0.0})

ax = fig.add_subplot(1, 3, 3, projection="3d")
t = np.linspace(-2.3, 2.3, 100)
s = np.linspace(-0.45, 0.45, 22)
T, S = np.meshgrid(t, s)
helix = np.stack([np.cos(T), np.sin(T), 0.32 * T], axis=-1)
tangent = np.stack([-np.sin(T), np.cos(T), 0.32 * np.ones_like(T)], axis=-1)
tangent = tangent / np.linalg.norm(tangent, axis=-1, keepdims=True)
Xdev = helix[..., 0] + S * tangent[..., 0]
Ydev = helix[..., 1] + S * tangent[..., 1]
Zdev = helix[..., 2] + S * tangent[..., 2]
ax.plot_surface(Xdev, Ydev, Zdev, color="#34d399", alpha=0.82, linewidth=0)
for idx in range(0, len(t), 12):
    base = np.array([np.cos(t[idx]), np.sin(t[idx]), 0.32 * t[idx]])
    tan = np.array([-np.sin(t[idx]), np.cos(t[idx]), 0.32])
    tan = tan / np.linalg.norm(tan)
    pts = np.array([base - 0.5 * tan, base + 0.5 * tan])
    ax.plot(pts[:, 0], pts[:, 1], pts[:, 2], color="#111827", lw=1.1)
ax.set_title("tangent developable: ruled flat patch")
ax.set_box_aspect((1, 1, 1.2))
flat_checks.append({"surface": "helix tangent developable away from edge", "principal_curvatures": [0.0, "nonzero transverse"], "K": 0.0})

for ax in fig.axes:
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_zticks([])
    ax.view_init(elev=22, azim=-55)
fig.suptitle("Flat surfaces can bend: one principal curvature vanishes along rulings", fontsize=14)
fig.tight_layout()
flat_path = save_matplotlib(fig, UNIT, "figures", "flat-developable-chapter-08.png", root=ARTIFACT_ROOT)
plt.close(fig)
flat_checks_path = save_json({"flat_surface_models": flat_checks}, UNIT, "checks", "flat-developable-checks.json", root=ARTIFACT_ROOT)
display_artifact(flat_path, width=920)

## 6. Constant Mean Curvature Via Parallel Surfaces

Parallel surfaces give the chapter a bridge between $H$ and $K$. If a surface is offset by signed distance $\lambda$ along the normal, then in a principal frame the new principal curvatures are

$$\kappa_i^{\lambda} = \frac{\kappa_i}{1 - \lambda\kappa_i}$$

as long as the denominator does not vanish. Therefore

$$K_{\lambda} = \frac{K}{1 - 2\lambda H + \lambda^2K}, \qquad
H_{\lambda} = \frac{H - \lambda K}{1 - 2\lambda H + \lambda^2K}.$$

The figure samples many curvature pairs on the constant mean curvature line $H=1/(2R)$. Offsetting by $\lambda=R$ sends every sampled pair to the same Gaussian curvature $1/R^2$, except where the parallel surface would be singular. This is the algebraic core of the constant-H/constant-positive-K bridge.

In [ ]:
R_symbol = sp.symbols("R", positive=True)
K_symbol = sp.symbols("K")
H_cmc = sp.Rational(1, 2) / R_symbol
parallel_K = sp.simplify(K_symbol / (1 - 2 * R_symbol * H_cmc + R_symbol**2 * K_symbol))
parallel_K_identity = sp.simplify(parallel_K - 1 / R_symbol**2)
assert parallel_K_identity == 0

R_value = 2.0
lambda_value = R_value
kappa1_values = np.linspace(-1.2, 1.65, 260)
kappa2_values = 1 / R_value - kappa1_values
K_values = kappa1_values * kappa2_values
denominator = 1 - 2 * lambda_value * (0.5 / R_value) + lambda_value**2 * K_values
valid = np.abs(denominator) > 1e-8
K_parallel = K_values[valid] / denominator[valid]
kp1 = kappa1_values[valid] / (1 - lambda_value * kappa1_values[valid])
kp2 = kappa2_values[valid] / (1 - lambda_value * kappa2_values[valid])

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
axes[0].plot(kappa1_values, kappa2_values, color="#111827", lw=2)
axes[0].axhline(0, color="#9ca3af", lw=1)
axes[0].axvline(0, color="#9ca3af", lw=1)
axes[0].set_xlabel("kappa_1")
axes[0].set_ylabel("kappa_2")
axes[0].set_title("constant H line: k1 + k2 = 1/R")
axes[0].grid(True, alpha=0.25)
axes[1].scatter(kp1, kp2, c=K_parallel, cmap="viridis", s=18)
axes[1].axhline(0, color="#9ca3af", lw=1)
axes[1].axvline(0, color="#9ca3af", lw=1)
axes[1].set_xlabel("parallel kappa_1")
axes[1].set_ylabel("parallel kappa_2")
axes[1].set_title("after offset lambda=R: K becomes 1/R^2")
axes[1].grid(True, alpha=0.25)
fig.suptitle("Parallel-surface transform linking constant mean curvature and constant positive K", fontsize=14)
fig.tight_layout()
mean_path = save_matplotlib(fig, UNIT, "figures", "mean-chapter-08.png", root=ARTIFACT_ROOT)
plt.close(fig)
parallel_checks = {
    "symbolic_identity_for_H_equal_1_over_2R": str(parallel_K_identity),
    "R": R_value,
    "lambda": lambda_value,
    "target_parallel_K": 1 / R_value**2,
    "sample_count": int(valid.sum()),
    "max_parallel_K_residual": float(np.max(np.abs(K_parallel - 1 / R_value**2))),
    "excluded_singular_samples": int((~valid).sum()),
}
parallel_checks_path = save_json(parallel_checks, UNIT, "checks", "parallel-mean-curvature-transform.json", root=ARTIFACT_ROOT)
display_artifact(mean_path, width=920)
parallel_checks

## 7. Compact Surfaces Must Have Some Positive Gaussian Curvature

The compactness result can be seen as a support-sphere proof. Put the origin somewhere in space and look for a point of the compact surface farthest from it. At that point the surface lies inside the sphere centered at the origin with the same radius. Any surface curve through the farthest point has second derivative constrained by the maximum of the squared-distance function. The normal curvatures therefore have the same sign and large enough magnitude, so their product is positive.

The sketch below shows a two-dimensional cross-section of the argument. The numerical check uses an ellipsoid with semiaxes $a>b>c$ at its farthest point $(a,0,0)$. The local graph has curvature magnitudes $a/b^2$ and $a/c^2$, so $K=a^2/(b^2c^2)>0$ and is at least the support-sphere value $1/a^2$.

In [ ]:
a_val, b_val, c_val = 3.0, 1.8, 1.2
support_radius = a_val
ellipsoid_K = a_val**2 / (b_val**2 * c_val**2)
support_K = 1 / support_radius**2
compact_checks = {
    "ellipsoid_semiaxes": {"a": a_val, "b": b_val, "c": c_val},
    "farthest_point": [a_val, 0.0, 0.0],
    "principal_curvature_magnitudes_at_farthest_point": [a_val / b_val**2, a_val / c_val**2],
    "ellipsoid_K_at_farthest_point": ellipsoid_K,
    "support_sphere_K": support_K,
    "positive_curvature_margin": ellipsoid_K - support_K,
}
assert compact_checks["ellipsoid_K_at_farthest_point"] > 0
assert compact_checks["positive_curvature_margin"] >= 0

fig, ax = plt.subplots(figsize=(8.2, 6.4))
angle = np.linspace(0, 2 * np.pi, 500)
ax.plot(support_radius * np.cos(angle), support_radius * np.sin(angle), color="#9ca3af", lw=2, label="support sphere cross-section")
ax.plot(a_val * np.cos(angle), b_val * np.sin(angle), color="#2563eb", lw=3, label="compact surface cross-section")
ax.scatter([a_val], [0], color="#dc2626", s=60, zorder=5)
ax.arrow(0, 0, a_val * 0.86, 0, head_width=0.08, head_length=0.15, fc="#111827", ec="#111827", length_includes_head=True)
ax.arrow(a_val, 0, 0.55, 0, head_width=0.08, head_length=0.15, fc="#dc2626", ec="#dc2626", length_includes_head=True)
ax.text(a_val + 0.1, 0.16, "farthest point", color="#dc2626")
ax.text(0.9, 0.15, "radius to support point", color="#111827")
ax.text(-2.65, -2.45, "surface lies inside the support sphere", color="#374151")
ax.set_aspect("equal", adjustable="box")
ax.set_xlabel("radial direction")
ax.set_ylabel("one tangent direction")
ax.set_title("Compact-surface support sphere: positive K occurs at a farthest point")
ax.legend(loc="upper left")
ax.grid(True, alpha=0.2)
compact_path = save_matplotlib(fig, UNIT, "figures", "compact-support-sphere-chapter-08.png", root=ARTIFACT_ROOT)
plt.close(fig)
compact_checks_path = save_json(compact_checks, UNIT, "checks", "compact-curvature-bound.json", root=ARTIFACT_ROOT)
display_artifact(compact_path, width=760)
compact_checks

## Applied Lab: Curvature Diagnostics You Can Modify

Use this lab table as a small checklist before changing parameters. Each row names a chapter claim, the computational object that represents it, and the residual that should stay small or the sign that should stay correct. The table is intentionally compact: it is meant to invite parameter changes, not to hide the geometry behind a large dataset.

In [ ]:
lab_rows = [
    {
        "chapter_claim": "K and H come from determinant and half-trace of W",
        "object_checked": "sample shape-operator matrices",
        "diagnostic": "max determinant/product and trace/mean residual",
        "value": max(shape_checks["max_det_product_residual"], shape_checks["max_trace_mean_residual"]),
        "expected": "0 within floating tolerance",
    },
    {
        "chapter_claim": "principal curvatures are extrema of normal curvature",
        "object_checked": "Euler normal-curvature samples",
        "diagnostic": "largest extrema residual",
        "value": max(row["extrema_residual"] for row in principal_checks),
        "expected": "near 0 after angular sampling",
    },
    {
        "chapter_claim": "constant Gaussian curvature solves K=-f''/f",
        "object_checked": "sphere, cylinder, negative-K revolution radius functions",
        "diagnostic": "symbolic residual flags",
        "value": int(all(row["symbolic_residual_is_zero"] for row in constant_k_checks)),
        "expected": "1 means every symbolic residual vanished",
    },
    {
        "chapter_claim": "flat developables can bend while K=0",
        "object_checked": "cylinder, cone, tangent developable models",
        "diagnostic": "recorded K values",
        "value": max(abs(float(row["K"])) for row in flat_checks),
        "expected": "0 for the displayed developable models",
    },
    {
        "chapter_claim": "parallel surfaces connect H=1/(2R) to K=1/R^2",
        "object_checked": "sampled principal-curvature pairs on a constant-H line",
        "diagnostic": "max transformed-K residual",
        "value": parallel_checks["max_parallel_K_residual"],
        "expected": "0 within floating tolerance away from singular offsets",
    },
    {
        "chapter_claim": "a compact surface has a point with K>0",
        "object_checked": "ellipsoid support-sphere sample",
        "diagnostic": "K minus support-sphere K",
        "value": compact_checks["positive_curvature_margin"],
        "expected": ">= 0 and ellipsoid K > 0",
    },
]
lab_table = pd.DataFrame(lab_rows)
lab_table_path = artifact_path(UNIT, "tables", "curvature-diagnostics.csv", root=ARTIFACT_ROOT)
lab_table.to_csv(lab_table_path, index=False)
lab_table

## Visual Storyboard Implemented

| Storyboard item | Artifact | Library route | Inspection target | Check |
| --- | --- | --- | --- | --- |
| Shape operator invariants | `checks/shape-operator-invariants.json` | SymPy plus NumPy | determinant, half-trace, characteristic equation | symbolic identities and sample residuals |
| Principal curvature extrema | `figures/principal-chapter-08.png` | Matplotlib polar plots | extrema of $\kappa_n(\theta)$ at principal directions | sampled extrema residuals |
| Curvature sign in 3D | `figures/curvature-sign-chapter-08.png` | Matplotlib 3D | elliptic, hyperbolic, parabolic, planar local models | stored $K,H$ signs |
| Constant Gaussian curvature | `figures/constant-k-chapter-08.png`, `interactive/constant-gaussian-curvature-surfaces.html` | SymPy plus Plotly 3D | $K=-f''/f$ and rotating surface models | symbolic ODE residuals |
| Flat surfaces | `figures/flat-developable-chapter-08.png` | Matplotlib 3D | ruled directions where one principal curvature vanishes | stored zero-$K$ checks |
| Constant mean curvature | `figures/mean-chapter-08.png` | SymPy plus Matplotlib | parallel-surface curvature transform | exact identity and numeric residual |
| Compact-surface constraint | `figures/compact-support-sphere-chapter-08.png` | Matplotlib proof sketch | farthest point and support sphere | ellipsoid bound $K >= 1/a^2 > 0$ |

In [ ]:
visual_storyboard = {
    "chapter_goal": "Extract Gaussian, mean, and principal curvatures from the shape operator and use them to inspect constant K, flat, constant H, and compact-surface behavior.",
    "source_span": SOURCE_SPAN,
    "library_routing": [
        {"concept": "shape operator identities", "library": "SymPy", "why": "exact determinant, trace, and characteristic-polynomial checks"},
        {"concept": "normal curvature extrema", "library": "Matplotlib", "why": "durable labeled polar figure for principal directions"},
        {"concept": "local surface type", "library": "Matplotlib 3D", "why": "static inspectable comparison of elliptic, hyperbolic, parabolic, and planar models"},
        {"concept": "constant Gaussian curvature surfaces", "library": "Plotly", "why": "rotatable 3D surface family stored as a standalone HTML artifact"},
        {"concept": "parallel-surface curvature transform", "library": "SymPy and Matplotlib", "why": "exact rational identity plus sampled visual bridge between H and K"},
    ],
    "visual_sequence": [
        str(principal_path.relative_to(UNIT_ARTIFACT_ROOT)),
        str(curvature_sign_path.relative_to(UNIT_ARTIFACT_ROOT)),
        str(constant_k_path.relative_to(UNIT_ARTIFACT_ROOT)),
        str(constant_k_html.relative_to(UNIT_ARTIFACT_ROOT)),
        str(flat_path.relative_to(UNIT_ARTIFACT_ROOT)),
        str(mean_path.relative_to(UNIT_ARTIFACT_ROOT)),
        str(compact_path.relative_to(UNIT_ARTIFACT_ROOT)),
    ],
    "computational_checks": [
        str(shape_checks_path.relative_to(UNIT_ARTIFACT_ROOT)),
        str(principal_checks_path.relative_to(UNIT_ARTIFACT_ROOT)),
        str(curvature_sign_checks_path.relative_to(UNIT_ARTIFACT_ROOT)),
        str(constant_k_checks_path.relative_to(UNIT_ARTIFACT_ROOT)),
        str(flat_checks_path.relative_to(UNIT_ARTIFACT_ROOT)),
        str(parallel_checks_path.relative_to(UNIT_ARTIFACT_ROOT)),
        str(compact_checks_path.relative_to(UNIT_ARTIFACT_ROOT)),
    ],
}
visual_storyboard_path = save_json(visual_storyboard, UNIT, "checks", "visual-storyboard.json", root=ARTIFACT_ROOT)
visual_storyboard

## final_sanity checks

The final cell checks three kinds of promises. First, the core identities used in the chapter have zero or tiny residuals. Second, every displayed artifact exists and is nonempty. Third, the saved table and JSON files contain the validation data needed to audit the notebook without rerunning every plot by hand.

In [ ]:
artifact_paths = [
    source_span_path,
    shape_checks_path,
    principal_checks_path,
    curvature_sign_checks_path,
    constant_k_checks_path,
    flat_checks_path,
    parallel_checks_path,
    compact_checks_path,
    visual_storyboard_path,
    lab_table_path,
    principal_path,
    curvature_sign_path,
    constant_k_path,
    constant_k_html,
    flat_path,
    mean_path,
    compact_path,
]

identity_residuals = {
    "shape_operator_det_trace": max(shape_checks["max_det_product_residual"], shape_checks["max_trace_mean_residual"]),
    "principal_extrema": max(row["extrema_residual"] for row in principal_checks),
    "constant_K_symbolic_failures": sum(not row["symbolic_residual_is_zero"] for row in constant_k_checks),
    "flat_K_abs_max": max(abs(float(row["K"])) for row in flat_checks),
    "parallel_surface_K_residual": parallel_checks["max_parallel_K_residual"],
    "compact_positive_margin": compact_checks["positive_curvature_margin"],
}

for path in artifact_paths:
    min_bytes = 64 if path.suffix.lower() in {".json", ".csv"} else 512
    assert_artifact(path, min_bytes=min_bytes)

assert identity_residuals["shape_operator_det_trace"] < 1e-12
assert identity_residuals["principal_extrema"] < 1e-4
assert identity_residuals["constant_K_symbolic_failures"] == 0
assert identity_residuals["flat_K_abs_max"] == 0
assert identity_residuals["parallel_surface_K_residual"] < 1e-10
assert identity_residuals["compact_positive_margin"] >= 0

final_sanity = {
    "unit": UNIT,
    "source_span": SOURCE_SPAN,
    "artifact_count": len(artifact_paths),
    "artifact_names": [str(path.relative_to(UNIT_ARTIFACT_ROOT)) for path in artifact_paths],
    "identity_residuals": identity_residuals,
    "lab_rows": len(lab_table),
    "generic_visual_builder_removed": True,
}
final_sanity_path = save_json(final_sanity, UNIT, "checks", "final-sanity.json", root=ARTIFACT_ROOT)
notebook_sanity_path = save_json(final_sanity, UNIT, "checks", "notebook-sanity.json", root=ARTIFACT_ROOT)
assert_artifact(final_sanity_path, min_bytes=512)
assert_artifact(notebook_sanity_path, min_bytes=512)
final_sanity

## Takeaways

The shape operator is the chapter's organizing object. Its determinant gives Gaussian curvature, its half-trace gives mean curvature, and its eigenvalues give the principal curvatures.

The signs of the principal curvatures explain the local surface type: same sign gives elliptic behavior, opposite signs give saddle behavior, and a zero factor gives flat or parabolic behavior.

Constant curvature conditions become inspectable when translated into formulas: $K=-f''/f$ for selected revolution models and rational principal-curvature transforms for parallel surfaces.

Flat surfaces need not be planes. They can bend as developable ruled pieces because one principal curvature vanishes.

Compactness adds a global restriction: a compact surface in three-space must have at least one point with positive Gaussian curvature, visible through the support-sphere argument.